In [ ]:
# ================================
# Stochastic SQP-PINN (JAX) - POINTWISE (SIGNED) constraints + optional "K steps per sample"
#
# Objective: data MSE minibatch from burgers.mat
#   - optional MC objective averaging (mc_obj_batches)
#
# Constraints: POINTWISE / BIN-WISE MEAN (SIGNED, not RMS)
#   PDE cell k:    c_pde[k]    = mean_cell[ r_pde(x,t) ]   (if N=1 => r at that point)
#   BC time-bin j: c_bc_val[j] = mean_bin[ uL-uR ]
#                 c_bc_der[j] = mean_bin[ uxL-uxR ]
#   IC x-bin j:    c_ic[j]     = mean_bin[ u(x,0)-u0(x) ]
#
# Stratified sampling (fixed # points per region/bin) -> unbiased per-bin mean.
#
# IMPORTANT knobs for "iterations per sample":
#   - hold_data_K : number of SQP steps to reuse the SAME data minibatch
#   - hold_con_K  : number of SQP steps to reuse the SAME constraint samples (Xf/XL/XR/Xic + ids)
#
# If you set hold_data_K = hold_con_K = 5000, the whole run is deterministic (given same seeds),
# because sampling happens only once.
#
# Scaling:
#   - Objective: fixed anchor scaling (computed once at theta0)
#   - Constraints: RMS+EMA scalar scaling on ||J||_rms (updated every iter)
#   - LM invariance: lam_eff = lam*(s_con^2)
#
# IMPORTANT: DO NOT CHANGE lambda bounds:
#   cal_d_and_y uses lam_min=1e-2, lam_max=1
# ================================

import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"  # set BEFORE importing jax

import time, math
from functools import partial
import numpy as np
import scipy.io

import jax
import jax.numpy as jnp
from jax import random, grad, vmap, hessian

# -----------------------------
# Precision
# -----------------------------
USE_X64 = False
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32
EPS = 1e-12

# -----------------------------
# Batch sizes / region structure (EDIT THESE)
# -----------------------------
B_DATA = 1000

# PDE cells
NX_PDE_MOM = 25
NT_PDE_MOM = 25
K_PDE = NX_PDE_MOM * NT_PDE_MOM
N_PDE_PER_CELL = 1
B_F_TOTAL = K_PDE * N_PDE_PER_CELL

# BC bins
K_BC = 100
N_BC_PER_BIN = 1
B_BC_TOTAL = K_BC * N_BC_PER_BIN

# IC bins
K_IC = 300
N_IC_PER_BIN = 1
B_IC_TOTAL = K_IC * N_IC_PER_BIN

# Total constraint dimension
M_CON = K_PDE + 2 * K_BC + K_IC


# -----------------------------
# 1) MLP
# -----------------------------
def init_mlp_params(key, layer_sizes):
    params = []
    keys = random.split(key, len(layer_sizes) - 1)
    for k, (m, n) in zip(keys, zip(layer_sizes[:-1], layer_sizes[1:])):
        W = random.normal(k, (m, n), dtype=DTYPE) * jnp.sqrt(DTYPE(2.0) / DTYPE(m))
        b = jnp.zeros((n,), dtype=DTYPE)
        params.append({"W": W, "b": b})
    return params


def mlp_apply(params, x):
    h = x
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            h = jnp.tanh(h)
    return h  # (N,1)


# -----------------------------
# 2) Flatten/unflatten
# -----------------------------
def flatten_params(params):
    flat_parts = []
    shapes = []
    for layer in params:
        W, b = layer["W"], layer["b"]
        flat_parts.append(W.reshape(-1))
        flat_parts.append(b.reshape(-1))
        shapes.append((W.shape, b.shape))
    theta = jnp.concatenate(flat_parts).astype(DTYPE)
    return theta, tuple(shapes)


def unflatten_params(theta, shapes):
    params = []
    idx = 0
    for W_shape, b_shape in shapes:
        W_size = math.prod(W_shape)
        b_size = math.prod(b_shape)
        W = theta[idx: idx + W_size].reshape(W_shape); idx += W_size
        b = theta[idx: idx + b_size].reshape(b_shape); idx += b_size
        params.append({"W": W, "b": b})
    return params


# -----------------------------
# 3) Load burgers.mat
# -----------------------------
def load_burgers_mat(path):
    d = scipy.io.loadmat(path)
    t = d["t"].squeeze()
    x = d["x"].squeeze()
    usol = d["usol"]  # (nt, nx)
    nu = float(np.array(d["nu"]).squeeze())
    return t, x, usol, nu


def build_full_data_points(t_np, x_np, usol_np):
    T, X = np.meshgrid(t_np, x_np, indexing="ij")
    X_data = np.stack([X.reshape(-1), T.reshape(-1)], 1)
    u_data = usol_np.reshape(-1)
    return X_data, u_data


# -----------------------------
# 4) Sampling utilities
# -----------------------------
def sample_data_batch(key, X_data, u_data, B):
    N = X_data.shape[0]
    idx = random.randint(key, (B,), 0, N)
    return X_data[idx], u_data[idx]


def segment_sum(values, segment_ids, num_segments):
    out = jnp.zeros((num_segments,), dtype=values.dtype)
    return out.at[segment_ids].add(values)


def sample_pde_stratified(key, x_min, x_max, t_min, t_max):
    dx = (DTYPE(x_max) - DTYPE(x_min)) / DTYPE(NX_PDE_MOM)
    dt = (DTYPE(t_max) - DTYPE(t_min)) / DTYPE(NT_PDE_MOM)

    it, ix = jnp.meshgrid(
        jnp.arange(NT_PDE_MOM),
        jnp.arange(NX_PDE_MOM),
        indexing="ij"
    )
    it = it.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)
    ids_cell = (it * NX_PDE_MOM + ix).astype(jnp.int32)

    x0 = DTYPE(x_min) + DTYPE(ix) * dx
    t0 = DTYPE(t_min) + DTYPE(it) * dt

    u = random.uniform(key, (K_PDE, N_PDE_PER_CELL, 2), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = x0[:, None] + u[:, :, 0] * dx
    ts = t0[:, None] + u[:, :, 1] * dt

    X_f = jnp.stack([xs, ts], axis=-1).reshape(-1, 2)
    ids = jnp.repeat(ids_cell, N_PDE_PER_CELL)
    return X_f, ids


def sample_bc_stratified(key, x_min, x_max, t_min, t_max):
    dt = (DTYPE(t_max) - DTYPE(t_min)) / DTYPE(K_BC)
    j = jnp.arange(K_BC, dtype=jnp.int32)
    t0 = DTYPE(t_min) + DTYPE(j) * dt

    u = random.uniform(key, (K_BC, N_BC_PER_BIN), minval=0.0, maxval=1.0, dtype=DTYPE)
    ts = (t0[:, None] + u * dt).reshape(-1, 1)
    ids = jnp.repeat(j, N_BC_PER_BIN)

    XL = jnp.concatenate([DTYPE(x_min) * jnp.ones_like(ts), ts], axis=1)
    XR = jnp.concatenate([DTYPE(x_max) * jnp.ones_like(ts), ts], axis=1)
    return XL, XR, ids


def sample_ic_stratified(key, x_min, x_max, t0, x_grid, u0_grid):
    dx = (DTYPE(x_max) - DTYPE(x_min)) / DTYPE(K_IC)
    j = jnp.arange(K_IC, dtype=jnp.int32)
    x0 = DTYPE(x_min) + DTYPE(j) * dx

    u = random.uniform(key, (K_IC, N_IC_PER_BIN), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = (x0[:, None] + u * dx).reshape(-1, 1)
    ids = jnp.repeat(j, N_IC_PER_BIN)

    Xic = jnp.concatenate([xs, DTYPE(t0) * jnp.ones_like(xs)], axis=1)
    u0 = jnp.interp(xs[:, 0], x_grid, u0_grid)
    return Xic, u0, ids

def jitter_points(key, X, x_min, x_max, t_min, t_max,
                  sigma_x, sigma_t,
                  kind="gaussian",
                  clip_k=None):
    """
    Jitter points X[:,0]=x and X[:,1]=t with fresh random noise each call.

    kind:
      - "gaussian": dx ~ N(0, sigma_x^2), dt ~ N(0, sigma_t^2)
      - "uniform":  dx ~ U[-sigma_x, +sigma_x], dt ~ U[-sigma_t, +sigma_t]

    clip_k:
      - if not None and kind=="gaussian", clip dx,dt to [-k*sigma, +k*sigma] before adding.
    Always clips final (x,t) to [x_min,x_max]×[t_min,t_max].
    """
    if (sigma_x <= 0.0) and (sigma_t <= 0.0):
        return X

    key, kx, kt = random.split(key, 3)

    if kind == "gaussian":
        dx = DTYPE(sigma_x) * random.normal(kx, (X.shape[0],), dtype=DTYPE)
        dt = DTYPE(sigma_t) * random.normal(kt, (X.shape[0],), dtype=DTYPE)
        if clip_k is not None:
            k = DTYPE(clip_k)
            if sigma_x > 0:
                dx = jnp.clip(dx, -k * DTYPE(sigma_x), +k * DTYPE(sigma_x))
            if sigma_t > 0:
                dt = jnp.clip(dt, -k * DTYPE(sigma_t), +k * DTYPE(sigma_t))

    elif kind == "uniform":
        dx = DTYPE(sigma_x) * (2.0 * random.uniform(kx, (X.shape[0],), dtype=DTYPE) - 1.0)
        dt = DTYPE(sigma_t) * (2.0 * random.uniform(kt, (X.shape[0],), dtype=DTYPE) - 1.0)

    else:
        raise ValueError("kind must be 'gaussian' or 'uniform'")

    x = jnp.clip(X[:, 0] + dx, DTYPE(x_min), DTYPE(x_max))
    t = jnp.clip(X[:, 1] + dt, DTYPE(t_min), DTYPE(t_max))
    return jnp.stack([x, t], axis=1)


# -----------------------------
# 5) PDE residual
# -----------------------------
def pde_residual_unscaled(params, X_f, nu):
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]

    u = vmap(u_fun)(X_f)
    du = vmap(grad(u_fun))(X_f)
    H = vmap(hessian(u_fun))(X_f)

    u_x = du[:, 0]
    u_t = du[:, 1]
    u_xx = H[:, 0, 0]
    return u_t + u * u_x - DTYPE(nu) * u_xx


# -----------------------------
# 6) Objective: data MSE + anchor scaling
# -----------------------------
def data_mse_unscaled(params, Xb, ub):
    pred = mlp_apply(params, Xb)[:, 0]
    return jnp.mean((pred - ub) ** 2)


@partial(jax.jit, static_argnames=("shapes",))
def F_and_g_data_scaled_batch(theta, shapes, Xb, ub, scale_obj):
    def obj_theta(th):
        params = unflatten_params(th, shapes)
        return data_mse_unscaled(params, Xb, ub)

    val, g = jax.value_and_grad(obj_theta)(theta)
    return scale_obj * val, scale_obj * g


def compute_scale_obj_data_anchor(theta0, shapes, X_anchor, u_anchor, target=100.0):
    def obj_theta(th):
        params = unflatten_params(th, shapes)
        return data_mse_unscaled(params, X_anchor, u_anchor)

    g0 = grad(obj_theta)(theta0)
    ginf = jnp.linalg.norm(g0, jnp.inf)
    return (DTYPE(target) / jnp.maximum(DTYPE(target), ginf + DTYPE(EPS))).astype(DTYPE)


# -----------------------------
# 7) BC helper u, ux
# -----------------------------
@jax.jit
def u_and_ux(params, X):
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]

    u = vmap(u_fun)(X)
    du = vmap(grad(u_fun))(X)
    ux = du[:, 0]
    return u, ux


# -----------------------------
# 8) POINTWISE/SIGNED (mean) constraints + Jacobian
#    With N_* = 1, these are literally pointwise residuals/mismatches.
# -----------------------------
@partial(jax.jit, static_argnames=("shapes",))
def constraint_vector_cells_unscaled(theta, shapes,
                                     X_f, ids_f,
                                     X_bc_L, X_bc_R, ids_bc,
                                     X_ic, u0_ic, ids_ic,
                                     nu):
    params = unflatten_params(theta, shapes)

    # PDE signed residual
    r = pde_residual_unscaled(params, X_f, nu)
    c_pde = segment_sum(r, ids_f, K_PDE) / DTYPE(N_PDE_PER_CELL)

    # BC signed differences
    uL, uxL = u_and_ux(params, X_bc_L)
    uR, uxR = u_and_ux(params, X_bc_R)
    dv = (uL - uR)
    dd = (uxL - uxR)
    c_bc_val = segment_sum(dv, ids_bc, K_BC) / DTYPE(N_BC_PER_BIN)
    c_bc_der = segment_sum(dd, ids_bc, K_BC) / DTYPE(N_BC_PER_BIN)

    # IC signed mismatch
    u_ic = mlp_apply(params, X_ic)[:, 0]
    di = (u_ic - u0_ic)
    c_ic = segment_sum(di, ids_ic, K_IC) / DTYPE(N_IC_PER_BIN)

    return jnp.concatenate([c_pde, c_bc_val, c_bc_der, c_ic], axis=0)


@partial(jax.jit, static_argnames=("shapes",))
def C_and_J_cells_unscaled(theta, shapes,
                           X_f, ids_f,
                           X_bc_L, X_bc_R, ids_bc,
                           X_ic, u0_ic, ids_ic,
                           nu):
    def c_fun(th):
        return constraint_vector_cells_unscaled(
            th, shapes, X_f, ids_f,
            X_bc_L, X_bc_R, ids_bc,
            X_ic, u0_ic, ids_ic,
            nu
        )

    def c_fun_aux(th):
        c = c_fun(th)
        return c, c

    J_un, c_un = jax.jacrev(c_fun_aux, has_aux=True)(theta)
    return c_un, J_un


# -----------------------------
# 9) RMS+EMA scaling (for Jacobian magnitude)
# -----------------------------
def ema_update(old, new, alpha):
    return (DTYPE(1.0) - DTYPE(alpha)) * old + DTYPE(alpha) * new


def scalar_scale_from_mag(mag, target):
    return DTYPE(target) / jnp.maximum(DTYPE(target), mag + DTYPE(EPS))


# -----------------------------
# 10) KKT solve (LM proxy), lam_eff = lam*(s^2)
#     DO NOT CHANGE lambda bounds
# -----------------------------
@jax.jit
def kkt_solve_once(H, Jac, Grad, Cons, lam_eff):
    n = H.shape[0]
    m = Cons.shape[0]
    I_m = jnp.eye(m, dtype=H.dtype)

    top = jnp.concatenate([H, Jac.T], axis=1)
    bottom = jnp.concatenate([Jac, -lam_eff * I_m], axis=1)
    KKT = jnp.concatenate([top, bottom], axis=0)

    rhs = -jnp.concatenate([Grad, Cons])
    sol = jnp.linalg.solve(KKT, rhs)
    d = sol[:n]
    y = sol[n:]
    return d, y


def cal_d_and_y(H, Jac, Grad, Cons, s_con,
               ridge_init=1e-6, eta_up=10.0, eta_down=0.25,
               lam_min=1e-3, lam_max=1, res_tol=1e-6):
    lam = float(jnp.clip(DTYPE(ridge_init), DTYPE(lam_min), DTYPE(lam_max)))
    s2 = float(s_con * s_con)

    for _ in range(12):
        lam_eff = DTYPE(lam * s2)
        d, y = kkt_solve_once(H, Jac, Grad, Cons, lam_eff)
        res_vec = jnp.concatenate([H @ d + Jac.T @ y + Grad, Jac @ d + Cons])
        res = float(jnp.linalg.norm(res_vec, 2))
        if res <= res_tol:
            lam_next = float(jnp.clip(DTYPE(lam) * DTYPE(eta_down), DTYPE(lam_min), DTYPE(lam_max)))
            return d, y, lam_next, res
        lam = float(jnp.clip(DTYPE(lam) * DTYPE(eta_up), DTYPE(lam_min), DTYPE(lam_max)))

    lam_eff = DTYPE(lam * s2)
    d, y = kkt_solve_once(H, Jac, Grad, Cons, lam_eff)
    res_vec = jnp.concatenate([H @ d + Jac.T @ y + Grad, Jac @ d + Cons])
    res = float(jnp.linalg.norm(res_vec, 2))
    return d, y, lam, res


# -----------------------------
# 11) tau/ksi/alpha (Zhou-style)
# -----------------------------
def cal_tau(H, d, sigma, tau_pre, eps_tau, g, c):
    denom = float(g @ d + 0.5 * (d @ (H @ d)))
    if denom <= 1e-12:
        tau_trial = float("inf")
    else:
        tau_trial = (1.0 - sigma) * float(jnp.linalg.norm(c, 1)) / denom

    if tau_pre <= tau_trial:
        return tau_pre
    return min(tau_trial, (1.0 - eps_tau) * tau_pre)


def cal_ksi(d, tau, ksi_old, eps_ksi, g, c):
    denom = tau * float(jnp.linalg.norm(d) ** 2)
    if denom <= 1e-18:
        return ksi_old

    top = -tau * float(g @ d) + float(jnp.linalg.norm(c, 1))
    ksi_trial = top / denom

    if ksi_old <= ksi_trial:
        return ksi_old
    return min(ksi_trial, (1.0 - eps_ksi) * ksi_old)


def phi(alpha, eta, beta, tau, g, d, c, L, Gamma):
    normC1 = float(jnp.linalg.norm(c, 1))
    gd = float(g @ d)
    d2 = float(d @ d)

    term1 = (eta - 1.0) * alpha * beta * (-tau * gd + normC1)
    term2 = (abs(1.0 - alpha) - 1.0 + alpha) * normC1
    term3 = 0.5 * (tau * L + Gamma) * (alpha ** 2) * d2
    return term1 + term2 + term3


def cal_alpha(d, eta, beta, ksi, tau, L, Gamma, theta_val, g, c):
    denom = (tau * L + Gamma)
    if denom <= 1e-12:
        return 0.0

    alpha_min = 2.0 * (1.0 - eta) * beta * ksi * tau / denom
    a = max(alpha_min, 0.0)

    while (phi(1.1 * a, eta, beta, tau, g, d, c, L, Gamma) < 0.0
           and (1.1 * a < alpha_min + theta_val * beta)):
        a *= 1.1
    return float(a)


# -----------------------------
# 12) Training loop with:
#     - hold_data_K steps per data minibatch
#     - hold_con_K  steps per constraint sample set
# -----------------------------
def train_sqp_cellwise_pointwise(theta0, shapes,
                                 X_data, u_data, x_grid, u0_grid, nu,
                                 x_min, x_max, t_min, t_max,
                                 scale_obj,
                                 con_scale_target=100.0, con_ema_alpha=0.05,
                                 L=100.0, Gamma=100.0,
                                 max_iters=5000, print_every=10,
                                 beta0=1.0, beta_shift=200.0, beta_power=0.8,
                                 alpha_cap=1.0,
                                 seed=0,
                                 mc_con_batches=1,
                                 mc_obj_batches=1,
                                 obj_resample_each_mc=False,
                                 hold_data_K=1,      # <-- iterations per DATA sample
                                 hold_con_K=1):      # <-- iterations per CONSTRAINT sample

    key = random.PRNGKey(seed)
    theta = theta0
    n = theta.shape[0]
    H = jnp.eye(n, dtype=DTYPE)

    eta, sigma = 0.25, 0.1
    eps_tau, eps_ksi = 1e-2, 1e-2
    theta_val = 10.0
    tau_k, ksi_k = 5.0, 1.0
    lam_prev = 1e-6

    Jmag_ema = DTYPE(0.0)
    s_con = DTYPE(1.0)

    # cached samples
    Xb = ub = None
    Xf = ids_f = None
    XL = XR = ids_bc = None
    Xic = u0ic = ids_ic = None

    t0_wall = time.time()
    for k in range(1, max_iters + 1):
        beta_k = float(min(1.0, beta0 * (beta_shift / (beta_shift + k)) ** beta_power))

        # -----------------------------
        # (A) DATA sampling (held for hold_data_K steps)
        # -----------------------------
        if (k == 1) or ((k - 1) % int(hold_data_K) == 0) or (Xb is None):
            key, k_data = random.split(key, 2)
            Xb, ub = sample_data_batch(k_data, X_data, u_data, B_DATA)

        # Objective MC average (optionally reuse same batch)
        S_obj = int(mc_obj_batches)
        obj_acc = DTYPE(0.0)
        obj2_acc = DTYPE(0.0)
        g_acc = jnp.zeros((n,), dtype=DTYPE)

        for _ in range(S_obj):
            if obj_resample_each_mc:
                key, k_data_i = random.split(key, 2)
                Xb_i, ub_i = sample_data_batch(k_data_i, X_data, u_data, B_DATA)
            else:
                Xb_i, ub_i = Xb, ub

            obj_i, g_i = F_and_g_data_scaled_batch(theta, shapes, Xb_i, ub_i, scale_obj)
            obj_acc = obj_acc + obj_i
            obj2_acc = obj2_acc + obj_i * obj_i
            g_acc = g_acc + g_i

        obj_s = obj_acc / DTYPE(S_obj)
        g_s = g_acc / DTYPE(S_obj)

        obj_var = jnp.maximum(obj2_acc / DTYPE(S_obj) - obj_s * obj_s, DTYPE(0.0))
        obj_std = float(jnp.sqrt(obj_var + DTYPE(EPS)))
        obj_std_mean = float(obj_std / jnp.sqrt(DTYPE(S_obj) + DTYPE(EPS)))

        # -----------------------------
        # (B) CONSTRAINT sampling (held for hold_con_K steps)
        # -----------------------------
        if (k == 1) or ((k - 1) % int(hold_con_K) == 0) or (Xf is None):
            key, k_f, k_bc, k_ic = random.split(key, 4)
            Xf, ids_f = sample_pde_stratified(k_f, x_min, x_max, t_min, t_max)
            XL, XR, ids_bc = sample_bc_stratified(k_bc, x_min, x_max, t_min, t_max)
            Xic, u0ic, ids_ic = sample_ic_stratified(k_ic, x_min, x_max, t_min, x_grid, u0_grid)

        # Constraints/J MC average (if you want, set mc_con_batches>1; by default 1)
        S_con = int(mc_con_batches)
        c_acc = jnp.zeros((M_CON,), dtype=DTYPE)
        J_acc = jnp.zeros((M_CON, n), dtype=DTYPE)

        # If S_con>1 and you want independent constraints inside the same iteration,
        # set hold_con_K=1 and S_con>1. Here, for simplicity, we resample inside MC only if S_con>1.
        c_list = []
        for s in range(S_con):
            if S_con == 1:
                Xf_s, ids_f_s = Xf, ids_f
                XL_s, XR_s, ids_bc_s = XL, XR, ids_bc
                Xic_s, u0ic_s, ids_ic_s = Xic, u0ic, ids_ic
            else:
                key, k_f_s, k_bc_s, k_ic_s = random.split(key, 4)
                Xf_s, ids_f_s = sample_pde_stratified(k_f_s, x_min, x_max, t_min, t_max)
                XL_s, XR_s, ids_bc_s = sample_bc_stratified(k_bc_s, x_min, x_max, t_min, t_max)
                Xic_s, u0ic_s, ids_ic_s = sample_ic_stratified(k_ic_s, x_min, x_max, t_min, x_grid, u0_grid)

            c_tmp, J_tmp = C_and_J_cells_unscaled(
                theta, shapes,
                Xf_s, ids_f_s,
                XL_s, XR_s, ids_bc_s,
                Xic_s, u0ic_s, ids_ic_s,
                nu
            )
            c_acc = c_acc + c_tmp
            J_acc = J_acc + J_tmp
            c_list.append(c_tmp)

        c_un = c_acc / DTYPE(S_con)
        J_un = J_acc / DTYPE(S_con)

        # -----------------------------
        # (C) RMS+EMA scaling on J magnitude
        # -----------------------------
        J_rms = jnp.sqrt(jnp.mean(J_un ** 2) + DTYPE(EPS))
        if k == 1 and float(Jmag_ema) == 0.0:
            Jmag_ema = J_rms
        else:
            Jmag_ema = ema_update(Jmag_ema, J_rms, con_ema_alpha)

        s_con = scalar_scale_from_mag(Jmag_ema, con_scale_target)
        s_con = jax.lax.stop_gradient(s_con)

        Cons = s_con * c_un
        Jac = s_con * J_un

        # -----------------------------
        # (D) KKT direction
        # -----------------------------
        d, y, lam_prev, kkt_res = cal_d_and_y(H, Jac, g_s, Cons, s_con, ridge_init=lam_prev)

        # -----------------------------
        # (E) Step size
        # -----------------------------
        dn = float(jnp.linalg.norm(d, jnp.inf))
        if dn > 1e-10:
            tau_k = cal_tau(H, d, sigma, tau_k, eps_tau, g_s, Cons)
            ksi_k = cal_ksi(d, tau_k, ksi_k, eps_ksi, g_s, Cons)
            alpha = cal_alpha(d, eta, beta_k, ksi_k, tau_k, L, Gamma, theta_val, g_s, Cons)
        else:
            alpha = 2.0 * (1.0 - eta) * beta_k * ksi_k * tau_k / (tau_k * L + Gamma)

        alpha = float(min(alpha, alpha_cap))
        theta = theta + DTYPE(alpha) * d

        # -----------------------------
        # Prints
        # -----------------------------
        if k % print_every == 0:
            feas = float(jnp.mean(Cons ** 2))
            station = float(jnp.linalg.norm(g_s + Jac.T @ y, jnp.inf))

            # group split (SIGNED constraints)
            c_pde = c_un[:K_PDE]
            c_bc_val = c_un[K_PDE: K_PDE + K_BC]
            c_bc_der = c_un[K_PDE + K_BC: K_PDE + 2 * K_BC]
            c_ic = c_un[K_PDE + 2 * K_BC:]

            def rms(x):    return float(jnp.sqrt(jnp.mean(x ** 2) + DTYPE(EPS)))
            def maxabs(x): return float(jnp.max(jnp.abs(x)))

            pde_rms, pde_max = rms(c_pde), maxabs(c_pde)
            bcV_rms, bcV_max = rms(c_bc_val), maxabs(c_bc_val)
            bcD_rms, bcD_max = rms(c_bc_der), maxabs(c_bc_der)
            ic_rms, ic_max = rms(c_ic), maxabs(c_ic)

            # pointwise RMS of raw quantities on CURRENT held constraint sample
            params_tmp = unflatten_params(theta, shapes)
            r_pts = pde_residual_unscaled(params_tmp, Xf, nu)
            uL_pts, uxL_pts = u_and_ux(params_tmp, XL)
            uR_pts, uxR_pts = u_and_ux(params_tmp, XR)
            dv_pts = (uL_pts - uR_pts)
            dd_pts = (uxL_pts - uxR_pts)
            di_pts = (mlp_apply(params_tmp, Xic)[:, 0] - u0ic)

            pde_pt_rms = float(jnp.sqrt(jnp.mean(r_pts ** 2) + DTYPE(EPS)))
            bcV_pt_rms = float(jnp.sqrt(jnp.mean(dv_pts ** 2) + DTYPE(EPS)))
            bcD_pt_rms = float(jnp.sqrt(jnp.mean(dd_pts ** 2) + DTYPE(EPS)))
            ic_pt_rms = float(jnp.sqrt(jnp.mean(di_pts ** 2) + DTYPE(EPS)))

            # If S_con>1, show constraint noise across the MC batches in this iteration
            if S_con > 1:
                c_stack = jnp.stack(c_list, axis=0)  # (S_con, M_CON)
                c_mean = jnp.mean(c_stack, axis=0)
                c_var = jnp.maximum(jnp.mean((c_stack - c_mean[None, :]) ** 2, axis=0), DTYPE(0.0))
                c_std = jnp.sqrt(c_var + DTYPE(EPS))
                c_std_mean = c_std / jnp.sqrt(DTYPE(S_con) + DTYPE(EPS))
                con_std_max = float(jnp.max(c_std))
                con_std_mean_max = float(jnp.max(c_std_mean))
            else:
                con_std_max = 0.0
                con_std_mean_max = 0.0

            print(
                f"[iter={k}] obj_s={float(obj_s):.3e} (obj_std={obj_std:.2e}, obj_std_mean={obj_std_mean:.2e}, Sobj={S_obj}) "
                f"feas={feas:.3e} station={station:.3e} "
                f"alpha={alpha:.2e} beta={beta_k:.2e} lam={lam_prev:.2e} kkt_res={kkt_res:.1e} s={float(s_con):.2e} "
                f"| constraints RMS/max: PDE=({pde_rms:.2e},{pde_max:.2e}) "
                f"BCv=({bcV_rms:.2e},{bcV_max:.2e}) "
                f"BCd=({bcD_rms:.2e},{bcD_max:.2e}) "
                f"IC=({ic_rms:.2e},{ic_max:.2e}) "
                f"| pointwise_rms(held): PDE={pde_pt_rms:.2e} BCv={bcV_pt_rms:.2e} BCd={bcD_pt_rms:.2e} IC={ic_pt_rms:.2e} "
                f"| mc_con={S_con} (std_max={con_std_max:.2e}, stdmean_max={con_std_mean_max:.2e}) "
                f"| hold_data_K={int(hold_data_K)} hold_con_K={int(hold_con_K)}"
            )

    print(f"[done] elapsed={time.time() - t0_wall:.2f}s")
    return theta


# -----------------------------
# 13) Eval helpers
# -----------------------------
def make_grid_points(x_np, t_np):
    T, X = np.meshgrid(t_np, x_np, indexing="ij")
    X_grid = np.stack([X.reshape(-1), T.reshape(-1)], axis=1)
    return X_grid, T.shape


@partial(jax.jit, static_argnames=("shapes",))
def predict_u(theta, shapes, X):
    params = unflatten_params(theta, shapes)
    return mlp_apply(params, X)[:, 0]


def eval_full_grid(theta, shapes, x_np, t_np, usol_np):
    X_grid_np, grid_shape = make_grid_points(x_np, t_np)
    X_grid = jnp.array(X_grid_np, dtype=DTYPE)
    u_pred = np.array(predict_u(theta, shapes, X_grid)).reshape(grid_shape)
    u_true = usol_np
    mse = float(np.mean((u_pred - u_true) ** 2))
    rel_l2 = float(np.linalg.norm(u_pred - u_true) / (np.linalg.norm(u_true) + 1e-12))
    return mse, rel_l2, u_pred


def plot_heatmaps(x_np, t_np, u_true, u_pred):
    import matplotlib.pyplot as plt
    err = u_pred - u_true

    plt.figure()
    plt.pcolormesh(x_np, t_np, u_true, shading="auto")
    plt.colorbar()
    plt.title("u_true")
    plt.xlabel("x")
    plt.ylabel("t")
    plt.show()

    plt.figure()
    plt.pcolormesh(x_np, t_np, u_pred, shading="auto")
    plt.colorbar()
    plt.title("u_pred")
    plt.xlabel("x")
    plt.ylabel("t")
    plt.show()

    plt.figure()
    plt.pcolormesh(x_np, t_np, err, shading="auto")
    plt.colorbar()
    plt.title("u_pred - u_true")
    plt.xlabel("x")
    plt.ylabel("t")
    plt.show()


# -----------------------------
# 14) main
# -----------------------------
def main():
    t_np, x_np, usol_np, nu = load_burgers_mat("data/burgers.mat")
    print("loaded burgers.mat:", "t", t_np.shape, "x", x_np.shape, "usol", usol_np.shape, "nu", nu)

    x_min, x_max = float(x_np.min()), float(x_np.max())
    t_min, t_max = float(t_np.min()), float(t_np.max())
    print(f"Domain: x in [{x_min}, {x_max}], t in [{t_min}, {t_max}]")

    X_data_np, u_data_np = build_full_data_points(t_np, x_np, usol_np)
    X_data = jnp.array(X_data_np, dtype=DTYPE)
    u_data = jnp.array(u_data_np, dtype=DTYPE)

    x_grid = jnp.array(x_np, dtype=DTYPE)
    u0_grid = jnp.array(usol_np[0, :], dtype=DTYPE)

    hidden_dim = 30
    num_hidden = 3
    layer_sizes = [2] + [hidden_dim] * num_hidden + [1]
    key = random.PRNGKey(0)
    key, key_params = random.split(key)
    params0 = init_mlp_params(key_params, layer_sizes)
    theta0, shapes = flatten_params(params0)

    # objective anchor scaling
    key_anchor = random.PRNGKey(123)
    X_anchor, u_anchor = sample_data_batch(key_anchor, X_data, u_data, B=512)
    scale_obj = compute_scale_obj_data_anchor(theta0, shapes, X_anchor, u_anchor, target=100.0)

    print("scale_obj (fixed anchor) =", float(scale_obj))
    print("Constraint dimension M_CON =", M_CON, " (= K_PDE + 2*K_BC + K_IC )")
    print(f"Total PDE samples per iter: {B_F_TOTAL} (= {K_PDE} cells * {N_PDE_PER_CELL} each)")
    print(f"Total BC samples  per iter: {B_BC_TOTAL} (= {K_BC} bins * {N_BC_PER_BIN} each)")
    print(f"Total IC samples  per iter: {B_IC_TOTAL} (= {K_IC} bins * {N_IC_PER_BIN} each)")

    theta_star = train_sqp_cellwise_pointwise(
        theta0, shapes,
        X_data, u_data, x_grid, u0_grid, nu,
        x_min=x_min, x_max=x_max, t_min=t_min, t_max=t_max,
        scale_obj=scale_obj,

        con_scale_target=100.0, con_ema_alpha=0.05,
        L=100.0, Gamma=100.0,

        max_iters=2000, print_every=5,

        beta0=1.0, beta_shift=200.0, beta_power=0.8,
        alpha_cap=1e1,

        seed=0,

        mc_con_batches=1,
        mc_obj_batches=1,
        obj_resample_each_mc=False,

        # ---- THIS is your "iterations per sample" control ----
        hold_data_K=2000,   # keep SAME data batch for all 5000 steps
        hold_con_K=2000     # keep SAME constraint points for all 5000 steps
    )

    np.save("theta_star_pointwise.npy", np.array(theta_star))
    print("Saved theta -> theta_star_pointwise.npy")

    mse, rel_l2, u_pred = eval_full_grid(theta_star, shapes, x_np, t_np, usol_np)
    print(f"[EVAL] full-grid MSE={mse:.3e}, relL2={rel_l2:.3e}")

    plot_heatmaps(x_np, t_np, usol_np, u_pred)


if __name__ == "__main__":
    main()


loaded burgers.mat: t (201,) x (512,) usol (201, 512) nu 0.003183098861837907
Domain: x in [-1.0, 1.0], t in [0.0, 1.0]
scale_obj (fixed anchor) = 1.0
Constraint dimension M_CON = 1125  (= K_PDE + 2*K_BC + K_IC )
Total PDE samples per iter: 625 (= 625 cells * 1 each)
Total BC samples  per iter: 100 (= 100 bins * 1 each)
Total IC samples  per iter: 300 (= 300 bins * 1 each)
